In [5]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("datascikhan/cars-dataset")

# print("Path to dataset files:", path)

In [2]:
import pandas as pd
import numpy as np

In [3]:
from pathlib import Path

csv_path = Path("cars_dataset.csv")
if not csv_path.is_file():
    csv_path = Path("dataset") / "cars_dataset.csv"

df = pd.read_csv(csv_path)
df.head()

,vehicle_id,listing_id,seller_id,dealer_id,make,model,model_variant,model_year,vehicle_year,vehicle_age,...,description_quality_score,vehicle_presentation_score,image_quality_score,listing_completeness_score,vehicle_risk_score,mechanical_risk_score,accident_risk_score,maintenance_risk_score,depreciation_risk_score,resale_risk_score
0,V00000001,L00000001,S00000001,NaN,BMW,i4,i4 RS,2020,2020,6,...,41.7,46.4,45.2,46.6,32.3,40.2,NaN,69.2,32.5,69.8
1,V00000002,L00000002,S00000002,D00000002,Chevrolet,Equinox,Equinox SE,2016,2016,10,...,35.5,46.6,44.0,67.9,49.8,63.6,36.2,100.0,NaN,26.4
2,V00000003,L00000003,S00000003,D00000003,Kia,Sportage,Sportage M Sport,2024,2024,2,...,53.5,58.2,62.8,78.5,33.3,0.6,12.3,NaN,51.4,6.6
3,V00000004,L00000004,S00000004,NaN,Hyundai,Ioniq 5,Ioniq 5 Sport,2015,2015,11,...,46.2,59.5,40.7,51.2,29.8,29.5,24.5,100.0,100.0,26.9
4,V00000005,L00000005,S00000005,D00000005,Honda,Accord,Accord GT,2022,2022,4,...,23.1,42.5,17.4,78.0,22.5,0.0,10.4,74.7,74.6,36.4


In [4]:
filtered_df = df[['brand_segment', 'body_type', 'fuel_type', 'engine_power_hp', 'vehicle_age', 'odometer_km', "number_of_previous_owners", "accident_history", "service_history_quality","overall_condition_score","urban_rural", "listing_price",'demand_category','days_on_market']].copy()
filtered_df.head()

,brand_segment,body_type,fuel_type,engine_power_hp,vehicle_age,odometer_km,number_of_previous_owners,accident_history,service_history_quality,overall_condition_score,urban_rural,listing_price,demand_category,days_on_market
0,Premium,Sedan,Electric,327.2,6,52122,2,0,Full History,71.7,Urban,29079.71,Medium,15.0
1,Mass Market,SUV,Petrol,266.9,10,121918,1,1,Full History,60.9,Suburban,2210.41,Medium,51.0
2,Mass Market,SUV,Petrol,259.8,2,21939,1,0,Partial History,92.1,Rural,16168.18,High,21.0
3,Mass Market,SUV,Electric,156.8,11,162555,1,1,Full History,50.3,Suburban,2982.87,Medium,16.0
4,Mass Market,Sedan,Petrol,243.8,4,55307,1,0,Full History,74.9,Rural,9866.65,Medium,45.0


## Step 1 · Audit the candidate variables

`filtered_df` is the candidate table. The model predicts the **listing price tier** from vehicle attributes and history. Keep `demand_category` and `days_on_market` in the audit, and leave them out of the initial model inputs because their availability at listing time has not been established.

Check the number of observed values, distinct values and missing values per column. An observed “No History” is a valid service-history category. A blank means the category is unknown and stays missing for the missing-data experiment.

In [5]:
audit = pd.DataFrame({
    "dtype": filtered_df.dtypes.astype(str),
    "observed": filtered_df.notna().sum(),
    "distinct": filtered_df.nunique(),
    "missing_pct": (100 * filtered_df.isna().mean()).round(2),
})
print(f"Source: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
display(audit)

Source: 100,000 rows x 437 columns


,dtype,observed,distinct,missing_pct
brand_segment,object,100000,2,0.0
body_type,object,100000,9,0.0
fuel_type,object,100000,5,0.0
engine_power_hp,float64,100000,5868,0.0
vehicle_age,int64,100000,16,0.0
odometer_km,int64,100000,71390,0.0
number_of_previous_owners,int64,100000,9,0.0
accident_history,int64,100000,2,0.0
service_history_quality,object,96000,3,4.0
overall_condition_score,float64,99000,847,1.0


## Split before learning the encoding

Shuffle once with seed 42 and assign 80% of rows to training, 10% to validation and 10% to testing. Fit the category vocabulary and quantile cut points on the training rows. Reuse the same rules for validation and testing. The `source_row` index identifies the original row when checking splits or masks.

In [6]:
SEED = 42
categorical_cols = [
    "brand_segment", "body_type", "fuel_type", "accident_history",
    "service_history_quality", "urban_rural",
]
fixed_cuts = {"vehicle_age": [2, 5, 10], "number_of_previous_owners": [0, 1, 2]}
quantile_bins = {
    "engine_power_hp": 3, "odometer_km": 3,
    "overall_condition_score": 3, "listing_price": 5,
}
model_df = filtered_df[categorical_cols + list(fixed_cuts) + list(quantile_bins)].copy()
model_df.index.name = "source_row"

body = model_df["body_type"]
model_df["body_type"] = body.mask(
    body.notna() & ~body.isin(["Sedan", "SUV", "Pickup"]), "Other"
)

row_order = np.random.default_rng(SEED).permutation(len(model_df))
split_rows = dict(zip(
    ("train", "val", "test"),
    np.split(row_order, [len(row_order) * 8 // 10, len(row_order) * 9 // 10]),
))
train_raw = model_df.iloc[split_rows["train"]]

## Encode variables as discrete states

Category codes are zero-based and follow the order recorded in the schema. Keep the five observed fuel categories. The age bins are 0–2, 3–5, 6–10 and 11+ years. Previous owners are grouped as 0, 1, 2 and 3+. Engine power, mileage and overall condition use training-set terciles. Listing price uses training-set quintiles, ordered from low to high. Repeated quantile boundaries are merged. Blank entries remain missing after encoding.

In [7]:
encoded = pd.DataFrame(index=model_df.index)
schema = {}

for col in categorical_cols:
    states = sorted(train_raw[col].dropna().unique().tolist())
    codes = model_df[col].map({state: i for i, state in enumerate(states)})
    unseen = model_df[col].notna() & codes.isna()
    if unseen.any():
        raise ValueError(f"{col} contains a category absent from training")
    encoded[col] = codes.astype("Int64")
    schema[col] = {"states": states}

for col in [*fixed_cuts, *quantile_bins]:
    if col in fixed_cuts:
        cuts = fixed_cuts[col]
    else:
        observed = train_raw[col].dropna()
        if observed.empty:
            raise ValueError(f"No observed training values for {col}")
        fractions = np.arange(1, quantile_bins[col]) / quantile_bins[col]
        cuts = sorted({
            value for value in observed.quantile(fractions).tolist()
            if value < observed.max()
        })
    edges = [-np.inf, *cuts, np.inf]
    encoded[col] = pd.cut(model_df[col], edges, labels=False).astype("Int64")
    schema[col] = {
        "states": list(range(len(edges) - 1)),
        "cut_points": cuts,
        "right_closed": True,
    }

assert encoded.isna().equals(model_df.isna())
splits = {name: encoded.iloc[rows].copy() for name, rows in split_rows.items()}
train_df, val_df, test_df = (splits[name] for name in ("train", "val", "test"))
assert pd.concat(splits.values()).index.is_unique
print({name: len(part) for name, part in splits.items()})
display(pd.DataFrame({
    "observed_train_states": train_df.nunique(),
    "train_missing_pct": (100 * train_df.isna().mean()).round(2),
}))

{'train': 80000, 'val': 10000, 'test': 10000}


,observed_train_states,train_missing_pct
brand_segment,2,0.00
body_type,4,0.00
fuel_type,5,0.00
accident_history,2,0.00
service_history_quality,3,3.95
urban_rural,3,0.00
vehicle_age,4,0.00
number_of_previous_owners,4,0.00
engine_power_hp,3,0.00
odometer_km,3,0.00


## Prepare an artificial missingness check for E2

The example below independently hides each observed cell in two training columns with 20% probability. The percentage refers to **additional masking among observed cells**. Reuse the same masked training data across methods. Keep the original data and Boolean mask to score recovery of values that were deliberately hidden.

In [8]:
def mcar_mask(frame, columns, rate, seed):
    if not 0 <= rate <= 1:
        raise ValueError("rate must be between 0 and 1")
    mask = pd.DataFrame(False, index=frame.index, columns=frame.columns)
    draws = np.random.default_rng(seed).random((len(frame), len(columns)))
    mask[columns] = (draws < rate) & frame[columns].notna().to_numpy()
    return frame.mask(mask), mask


mask_columns = ["service_history_quality", "overall_condition_score"]
masked_train, newly_hidden = mcar_mask(train_df, mask_columns, rate=0.20, seed=SEED)
assert masked_train.isna().equals(train_df.isna() | newly_hidden)
display((newly_hidden[mask_columns].sum() / train_df[mask_columns].count()).round(3))

service_history_quality    0.200
overall_condition_score    0.198
dtype: float64

## Export the model inputs

Set `EXPORT = True` and run this cell to save the three splits, the column audit, and the encoding schema in `dataset/prepared/`. Load an exported split with `pd.read_csv(path, index_col="source_row", dtype="Int64")`. Use the schema to keep state order consistent when fitting conditional probability tables and running inference.

In [15]:
import json
from pathlib import Path

EXPORT = False
if EXPORT:
    csv_path = Path("cars_dataset.csv")
    output_dir = csv_path.parent / "prepared"
    output_dir.mkdir(exist_ok=True)
    for name, part in splits.items():
        part.to_csv(output_dir / f"{name}.csv")
    audit.to_csv(output_dir / "column_audit.csv", index_label="column")
    (output_dir / "schema.json").write_text(
        json.dumps({"seed": SEED, "columns": schema}, indent=2),
        encoding="utf-8",
    )
    print(f"Saved files to {output_dir.resolve()}")